# 1. Importing dependencies

In [ ]:
import numpy as np
import random
from copy import deepcopy

# 2. Parameters

In [ ]:
POPULATION_SIZE = 30
MAX_GENERATIONS = 20
MUTATION_PROB = 0.4
CROSSOVER_PROB = 0.6
MAX_TREE_DEPTH = 6

# Allowed mathematical operators
OPERATORS = [np.add, np.subtract, np.multiply, np.divide]


# 3. Expression Tree 

In [ ]:
class Node:
    def __init__(self, value, children=None):
        self.value = value
        self.children = children if children else []

    def __call__(self, **kwargs):
        try:
            if callable(self.value):
                return self.value(*[child(**kwargs) for child in self.children])
            return kwargs.get(self.value, self.value)
        except Exception as e:
            print(f"Error evaluating node {self.value}: {e}")
            return 0  # Fallback value

    def get_random_subtree(self):
        if not self.children or random.random() < 0.3:
            return self
        return random.choice(self.children).get_random_subtree()

    def set_random_subtree(self, new_subtree):
        if not self.children or random.random() < 0.3:
            self.value = new_subtree.value
            self.children = deepcopy(new_subtree.children)
        else:
            random.choice(self.children).set_random_subtree(new_subtree)

    def to_numpy(self):
        """Converts the expression tree to a valid NumPy expression"""
        if callable(self.value):
            if self.value == np.divide:
                return f"np.divide({self.children[0].to_numpy()}, np.clip({self.children[1].to_numpy()}, 1e-3, None))"
            return f"np.{self.value.__name__}({', '.join(child.to_numpy() for child in self.children)})"
        return str(self.value)


# 4. Defining an Individual

In [ ]:
class Individual:
    def __init__(self, tree):
        self.tree = tree
        self.fitness = float('inf')

    def evaluate(self, x):
        try:
            result = self.tree(**{f'x{i}': x[i] for i in range(len(x))})

            # Handle invalid values
            if np.isnan(result) or np.isinf(result):
                return 0  # Fallback value

            return result
        except Exception as e:
            print(f"Error in evaluate(): {e}")
            return 0  # Fallback value

    def compute_fitness(self, x, y):
        predictions = np.array([self.evaluate(xi) for xi in x.T])
        valid_mask = ~np.isnan(predictions) & ~np.isinf(predictions)

        if not valid_mask.any():
            self.fitness = float('inf')
            return self.fitness

        mse = np.mean((y[valid_mask] - predictions[valid_mask]) ** 2)
        self.fitness = mse
        return mse


# 5. Genetic Programming

In [ ]:
class GeneticProgram:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.population = [Individual(self.generate_random_tree()) for _ in range(POPULATION_SIZE)]

    def generate_random_tree(self, depth=0):
        if depth >= MAX_TREE_DEPTH or random.random() < 0.3:
            return Node(random.choice([random.uniform(-10, 10), f'x{random.randint(0, self.x.shape[0] - 1)}']))

        operation = random.choice(OPERATORS)
        return Node(operation, [self.generate_random_tree(depth+1), self.generate_random_tree(depth+1)])

    def select_parent(self):
        return min(random.sample(self.population, 5), key=lambda ind: ind.fitness)

    def crossover(self, parent1, parent2):
        child_tree = deepcopy(parent1.tree)
        child_tree.set_random_subtree(deepcopy(parent2.tree.get_random_subtree()))
        return Individual(child_tree)

    def mutate(self, individual):
        new_tree = deepcopy(individual.tree)
        new_tree.set_random_subtree(self.generate_random_tree())
        return Individual(new_tree)

    def evolve(self):
        for generation in range(MAX_GENERATIONS):
            print(f"Generation {generation+1}...")

            for individual in self.population:
                individual.compute_fitness(self.x, self.y)

            self.population.sort(key=lambda ind: ind.fitness)

            if generation % 5 == 0:
                print(f"Generation {generation+1} - Best fitness: {self.population[0].fitness}")

            new_population = self.population[:POPULATION_SIZE // 3]

            while len(new_population) < POPULATION_SIZE:
                if random.random() < CROSSOVER_PROB:
                    parent1, parent2 = self.select_parent(), self.select_parent()
                    new_population.append(self.crossover(parent1, parent2))
                else:
                    parent = self.select_parent()
                    new_population.append(self.mutate(parent))

            self.population = new_population

    def get_best_solution(self):
        return min(self.population, key=lambda ind: ind.fitness)


# 6. Loading Datasets

In [ ]:
datasets = {}
for i in range(1, 9):
    data = np.load(f"problem_{i}.npz")
    datasets[i] = (data["x"], data["y"])

# 7. Saving the best expression

In [ ]:
best_expressions = {}
for i in range(1, 9):
    x_train, y_train = datasets[i]

    print(f"Optimizing dataset {i}...")

    gp = GeneticProgram(x_train, y_train)
    gp.evolve()

    best_solution = gp.get_best_solution()
    best_expressions[i] = best_solution.tree.to_numpy()

    print(f"Dataset {i} optimized! Best fitness: {best_solution.fitness}")

with open("s335017.py", "w") as f:
    f.write("import numpy as np\n\n")
    for i in range(1, 9):
        f.write(f"def f{i}(x: np.ndarray) -> np.ndarray:\n")
        f.write(f"    return {best_expressions[i]}\n\n")

print("Optimization completed! Best expressions saved in s335017.py.")


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=f1ffb741-3d6c-4355-8041-11e3fb3b96ba' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>